注意：运行下面的代码首先要执行 [gen_demo_factor_data.py](../tools/gen_demo_factor_data.py) 脚本生成示例数据，并且根据 [JYDB](JYDB.ipynb#配置与连接) 设置好了聚源数据库配置文件。

In [ ]:
import datetime as dt
import warnings
warnings.filterwarnings('ignore')
import logging

import numpy as np

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

from QuantStudio.Tools.Visualization import qs_help

# 因子框架快速入门

本文档通过一个完整示例演示因子框架最核心的使用流程：**连接数据 → 获取因子 → 衍生计算 → 读取结果**。涉及的概念和 API 的详细说明请参见对应文档。

## 整体架构（30 秒了解）

```
FactorDB（因子库） → getTable() → FactorTable（因子表） → getFactor() → Factor（因子）
                                                                        ↓
                                                               Operator(Factor) → 衍生因子
```

详细架构和概念说明请参见 **[基本框架](基本框架.ipynb)**。

# 1. 连接因子库

QuantStudio 支持多种因子库，这里分别用 HDF5DB 和 JYDB 演示。更多因子库类型请参见 **[基本框架](基本框架.ipynb#可用因子库)**。

In [2]:
# 本地 HDF5 因子库
from QuantStudio.Factor.HDF5DB import HDF5DB

HDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()
print(f"HDF5DB 表列表: {HDB.TableNames}")

HDF5DB 表列表: ['index_cn_day_bar', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']


In [3]:
# 聚源数据库因子库
from QuantStudio.Factor.JYDB import JYDB

SDB = JYDB().connect()
print(f"JYDB 表列表（前5个）: {SDB.TableNames[:5]}")

JYDB 表列表（前5个）: ['交易日表(新)', '人员表', '行业表', '行业类别表', 'A股证券主表']


# 2. 获取基础因子

通过 `getTable` → `getFactor` 获取基础因子。基础因子是计算图的叶子节点，数据来源于底层数据库。

In [4]:
# 从 HDF5DB 获取因子
FT = HDB.getTable("stock_cn_day_bar")
Close = FT.getFactor("close")
High, Low = FT.getFactor("high"), FT.getFactor("low")

print(f"因子名称: {Close.Name}")
print(f"数据类型: {Close.getMetaData(key='DataType')}")

因子名称: close
数据类型: double


In [5]:
# 从 JYDB 获取因子
FT = SDB.getTable("日行情表", args={"LookBack": 0})
S_Close = FT.getFactor("收盘价(元)")
S_Open = FT.getFactor("今开盘(元)")

# 财务数据因子表支持丰富的计算模式
FT = SDB.getTable("资产负债表_新会计准则", args={"CalcType": "最新"})
Equity = FT.getFactor("归属母公司股东权益合计")

print(f"收盘价: {S_Close.Name}")
print(f"股东权益: {Equity.Name}")

收盘价: 收盘价(元)
股东权益: 归属母公司股东权益合计


# 3. 读取因子数据

调用 `readData(ids, dts)` 即可读取因子的时序-截面数据，返回 `DataFrame`。

> 因子表也可以直接 `readData`，返回的 `Panel` 是 DataFrames 的集合。详见 **[基本框架](基本框架.ipynb)**。

In [6]:
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

print("Close 因子数据:")
print(Close.readData(ids=IDs, dts=DTs))

Close 因子数据:
            000001.SZ  000002.SZ
2025-01-01   8.115185   4.760840
2025-01-02   0.352198   1.806606
2025-01-03   6.019437   0.633690
2025-01-04   3.936297   3.755494
2025-01-05   4.884425   1.342670


# 4. 定义衍生因子

衍生因子由**算子（`FactorOperator`）**作用于描述子（依赖因子）产生。QuantStudio 支持四种运算类型：

| 运算 | 算子类 | 典型场景 |
|------|--------|----------|
| 单点运算 | `PointOperator` | 估值指标：PB = 总市值 / 股东权益 |
| 时序运算 | `TimeOperator` | 移动平均线、EMA |
| 截面运算 | `SectionOperator` | 标准化、截面排名 |
| 面板运算 | `PanelOperator` | 双重标准化 |

详细说明请参见 **[因子开发](因子开发.ipynb)**。

## 4.1 表达式方式（单点运算）

最简单的因子定义方式，直接使用 Python 运算符。本质上是 `BasicOperator` 提供的单点运算快捷方式。

In [7]:
from QuantStudio.Factor.BasicOperator import rename

# 用表达式计算中位数价格
Mid = rename((High + Low) / 2, factor_name="Mid")

# JYDB 因子之间的运算
FromS = rename(S_Open * 10000 / Equity, factor_name="市值开盘比")

print(f"衍生因子: {Mid.Name}")
print(Mid.readData(ids=IDs, dts=DTs))

衍生因子: Mid
            000001.SZ  000002.SZ
2025-01-01   6.801660   4.476960
2025-01-02   5.069191   4.899096
2025-01-03   4.807258   2.502005
2025-01-04   2.762997   2.429622
2025-01-05   4.032128   4.843444


## 4.2 工厂函数方式（时序运算）

使用 `makeFactorOperator` 创建自定义算子。时序运算需要指定 `LookBack`（回溯期数）。

In [8]:
from QuantStudio.Factor.FactorOperation import makeFactorOperator

# 定义移动平均算子
def MAFunc(f, idt, iid, x, args):
    return np.mean(x[0], axis=0)

calcMA = makeFactorOperator(MAFunc, operator_type="Time", 
    args={"Arity": 1, "DTMode": "单时点", "IDMode": "多ID", "LookBack": [4]})

MA5 = calcMA(Close, factor_args={"Name": "MA5"})
print(f"5日均线: {MA5.Name}")

DTRuler = list(reversed([dt.datetime(2025, 1, 1) - dt.timedelta(i+1) for i in range(10)])) + DTs
print(MA5.readData(ids=IDs, dts=DTs, dt_ruler=DTRuler))

5日均线: MA5
            000001.SZ  000002.SZ
2025-01-01        NaN        NaN
2025-01-02        NaN        NaN
2025-01-03        NaN        NaN
2025-01-04        NaN        NaN
2025-01-05   4.661508    2.45986


## 4.3 装饰器方式（截面运算）

使用 `@FactorOperatorized` 装饰器是更简洁的算子定义方式。截面运算处理同时点全部证券的数据。

In [9]:
from QuantStudio.Factor.FactorOperation import FactorOperatorized

@FactorOperatorized(operator_type="Section", args={"Arity": 1, "DTMode": "多时点"})
def calcZScore(f, idt, iid, x, args):
    return ((x[0].T - np.nanmean(x[0], axis=1)) / np.nanstd(x[0], axis=1)).T

Close_ZScore = calcZScore(Close, factor_args={"Name": "Close_ZScore"})
print(f"收盘价 Z-Score: {Close_ZScore.Name}")
print(Close_ZScore.readData(ids=IDs, dts=DTs))

收盘价 Z-Score: Close_ZScore
            000001.SZ  000002.SZ
2025-01-01        1.0       -1.0
2025-01-02       -1.0        1.0
2025-01-03        1.0       -1.0
2025-01-04        1.0       -1.0
2025-01-05        1.0       -1.0


## 4.4 使用内置算子

QuantStudio 预定义了 30+ 个常用算子，可直接使用。完整列表请参见 **[因子开发](因子开发.ipynb#内置算子)**。

In [10]:
from QuantStudio.Factor.FactorOperator import Log, RollingRank

# 取对数
LogClose = Log(base=np.e)(Close, factor_args={"Name": "LogClose"})

print("对数收盘价:")
print(LogClose.readData(ids=IDs, dts=DTs))

对数收盘价:
            000001.SZ  000002.SZ
2025-01-01   2.093737   1.560424
2025-01-02  -1.043562   0.591450
2025-01-03   1.794994  -0.456195
2025-01-04   1.370241   1.323220
2025-01-05   1.586052   0.294660


# 5. 用计算图引擎执行

当需要批量计算多个因子并利用缓存时，可以将因子集合交给引擎执行。引擎负责遍历 DAG、调度节点、管理缓存。

详细说明请参见 **[计算图框架](../Core/计算图框架.ipynb)** 和 **[计算引擎](../Core/计算引擎.ipynb)**。

In [11]:
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Core.CalcEngine import Engine

# 定义一组因子
Factors = [Close, Mid, MA5, Close_ZScore]

# 创建上下文和引擎
DTRuler = list(reversed([dt.datetime(2025, 1, 1) - dt.timedelta(i+1) for i in range(10)])) + DTs
SectionIDs = Close.getID()
Context = FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs)
ExecEngine = Engine()

# 执行计算
from QuantStudio.Factor.Factor import FactorLocalContext

Rslt = ExecEngine.run(
    Factors, Context,
    fwd_data_list=[FactorLocalContext(DTs=DTs, IDs=IDs, SectionIDs=SectionIDs)] * len(Factors)
)

# 查看结果
for i, iFactor in enumerate(Factors):
    print(f"{iFactor.Name}:")
    print(Rslt[i])
    print()

close:
            000001.SZ  000002.SZ
2025-01-01   8.115185   4.760840
2025-01-02   0.352198   1.806606
2025-01-03   6.019437   0.633690
2025-01-04   3.936297   3.755494
2025-01-05   4.884425   1.342670

Mid:
            000001.SZ  000002.SZ
2025-01-01   6.801660   4.476960
2025-01-02   5.069191   4.899096
2025-01-03   4.807258   2.502005
2025-01-04   2.762997   2.429622
2025-01-05   4.032128   4.843444

MA5:
            000001.SZ  000002.SZ
2025-01-01        NaN        NaN
2025-01-02        NaN        NaN
2025-01-03        NaN        NaN
2025-01-04        NaN        NaN
2025-01-05   4.661508    2.45986

Close_ZScore:
            000001.SZ  000002.SZ
2025-01-01   1.708604   0.312126
2025-01-02  -1.459167  -0.994224
2025-01-03   0.403303  -1.259084
2025-01-04  -0.495191  -0.575807
2025-01-05  -0.131889  -1.538464



# 6. 写入 HDF5DB 持久化

计算好的衍生因子可以通过 `writeData` 写入 HDF5DB 持久化保存。也可以使用 `FactorStorer` 节点在引擎执行时自动写入。详见 **[HDF5DB](HDF5DB.ipynb)**。

In [12]:
from QuantStudio.Core.QSObject import Panel

# 将引擎计算得到的结果组装成 Panel 写入
Data = Panel({Factors[i].Name: Rslt[i] for i in range(len(Factors))})
HDB.writeData(data=Data, table_name="my_factors", if_exists="update",
              data_type={f.Name: f.getMetaData(key="DataType") for f in Factors})

print(f"写入后的表列表: {HDB.TableNames}")
print(f"my_factors 因子列表: {HDB.getTable('my_factors').FactorNames}")

写入后的表列表: ['index_cn_day_bar', 'my_factors', 'stock_cn_day_bar', 'stock_cn_factor_value', 'stock_cn_industry', 'stock_cn_status']
my_factors 因子列表: ['Close_ZScore', 'MA5', 'Mid', 'close']


In [13]:
# 清理测试数据
HDB.deleteTable("my_factors")

0

# 下一步

| 想了解什么 | 推荐文档 |
|------------|----------|
| 因子框架的架构和 API 总览 | **[基本框架](基本框架.ipynb)** |
| 四种运算的详细参数和用法 | **[因子开发](因子开发.ipynb)** |
| 聚源数据库的使用 | **[JYDB](JYDB.ipynb)** |
| 本地 HDF5 文件存储 | **[HDF5DB](HDF5DB.ipynb)** |
| 计算图原理和引擎调度 | **[计算图框架](../Core/计算图框架.ipynb)** |
| 不同引擎的并行策略 | **[计算引擎](../Core/计算引擎.ipynb)** |